# CharacterTextSplitter 실습

긴 텍스트를 하나의 구분자(separator) 기준으로 잘라서 지정한 글자 수 단위의 조각(chunk)으로 나누는 `CharacterTextSplitter`를 실습한다. 예시 데이터로는 AI 용어를 정의해둔 한글 용어집(`data/appendix-keywords.txt`)을 사용한다. 이 노트북은 텍스트 분할만 다루기 때문에 LLM API를 호출하지 않고, 그래서 `.env`나 API 키 로딩도 필요 없다.

In [1]:
# 용어집 텍스트 파일을 UTF-8 인코딩으로 읽어온다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

## 1. 원본 텍스트 미리보기

읽어온 텍스트의 앞부분 500자만 출력해서 어떤 내용인지 확인한다. "Semantic Search", "Embedding", "Token" 같은 AI 용어들이 각각 정의·예시·연관키워드 형식으로 정리되어 있고, 용어 사이는 빈 줄(`\n\n`)로 구분되어 있다.

In [2]:
print(file[:500])

Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어


## 2. CharacterTextSplitter 설정

`CharacterTextSplitter`는 지정한 `separator`를 기준으로 텍스트를 나누는 가장 단순한 형태의 텍스트 분할기다.

- `separator="\n\n"` : 빈 줄(문단 구분)을 기준으로 우선 나눈다.
- `chunk_size=210` : 조각 하나의 최대 글자 수.
- `chunk_overlap=0` : 조각들 사이에 겹치는 부분 없음.
- `length_function=len` : 글자 수를 셀 때 사용할 함수(기본 `len`).

In [3]:
from langchain_text_splitters import CharacterTextSplitter

# 빈 줄을 구분자로 삼아 최대 210자 단위로 텍스트를 나누는 분할기를 만든다.
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=210,
    chunk_overlap=0,
    length_function=len,
)

## 3. create_documents()로 Document 리스트 만들기

`create_documents()`에 텍스트 리스트를 넘기면, 각 텍스트를 위 설정대로 나눈 뒤 조각마다 `Document` 객체로 감싸서 반환한다. 첫 번째 조각(`texts[0]`)의 길이와 내용을 확인해보면, `chunk_size=210`이 넘지 않는 선에서 `\n\n` 구분자를 기준으로 "Semantic Search" 항목 전체를 담고, 다음 항목("Embedding")의 제목까지만 포함한 뒤 잘린 것을 볼 수 있다(실제 글자 수는 197자로 210자 이내).

In [4]:
# 텍스트를 하나의 Document 리스트로 만든다.
texts = text_splitter.create_documents([file])
print(len(texts[0].page_content))
print(texts[0])

197
page_content='Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding'


## 4. 여러 문서 + 문서별 metadata 한 번에 넣기

`create_documents()`는 텍스트를 여러 개 리스트로 넘길 수도 있고, 이때 `metadatas` 인자로 각 원본 텍스트에 대응하는 메타데이터를 함께 지정할 수 있다. 여기서는 같은 `file` 텍스트를 두 번 넣으면서, 각각 `{"document": 1}`, `{"document": 2}`라는 메타데이터를 붙였다. 첫 번째 조각(`documents[0]`)을 보면 내용은 3번과 동일하지만 `metadata={'document': 1}`이 함께 붙어 있는 것을 확인할 수 있다.

In [5]:
# file을 두 번 넣고, 각 원본 문서에 대응하는 metadata를 지정한다.
metadatas = [
    {"document": 1},
    {"document": 2},
]

documents = text_splitter.create_documents(
    [
        file,
        file
    ],
    metadatas=metadatas
)

print(documents[0])

page_content='Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding' metadata={'document': 1}


## 5. 전체 조각 개수 확인

`file` 하나를 나누면 32개의 조각이 나온다(3번에서 첫 조각 크기가 210자에 가까운 걸 보면 대략 12,533자 ÷ 210자 ≈ 32개인 것과 맞아떨어진다). 같은 텍스트를 두 번 넣었으니 총 조각 개수는 32 × 2 = 64개가 된다.

In [6]:
len(documents)

64

## 6. metadata가 조각마다 올바르게 유지되는지 확인

`documents[1]`은 첫 번째 원본 문서(`file` 첫 번째 사본)에서 나온 두 번째 조각이다. `metadata`를 확인해보면 `{'document': 1}`로, 원본 문서가 여러 조각으로 나뉘어도 그 원본에 지정했던 metadata가 모든 조각에 그대로 상속되는 것을 알 수 있다.

In [7]:
documents[1].metadata

{'document': 1}

## 7. split_text(): Document가 아닌 순수 문자열로 받기

`create_documents()`가 `Document` 객체 리스트를 반환하는 것과 달리, `split_text()`는 나눈 결과를 `Document`로 감싸지 않고 순수 문자열 리스트로 반환한다. metadata가 필요 없이 텍스트 조각 자체만 필요할 때 더 가볍게 쓸 수 있는 저수준(low-level) 메서드다. 첫 번째 조각(`[0]`)은 3번의 `texts[0].page_content`와 동일한 내용이다.

In [8]:
text_splitter.split_text(file)[0]

'Semantic Search\n\n정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.\n예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.\n연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝\n\nEmbedding'

## 정리

| 메서드 | 반환 타입 | metadata 지정 |
|---|---|---|
| `create_documents(texts, metadatas=...)` | `Document` 객체 리스트 | 원본 문서별로 지정 가능, 모든 조각에 상속 |
| `split_text(text)` | 문자열 리스트 | 불가(순수 텍스트만) |

`CharacterTextSplitter`는 하나의 `separator`만 기준으로 나누기 때문에, 그 구분자로 나눈 조각이 `chunk_size`보다 크면 더 잘게 쪼개지 않고 그대로 둔다(반면 `RecursiveCharacterTextSplitter`는 여러 구분자를 순서대로 시도하며 최대한 `chunk_size`에 맞춘다). 여러 문서를 한 번에 나눌 때는 `metadatas`로 원본 출처를 함께 기록해두면, 나뉜 조각이 어느 원본에서 왔는지 나중에도 추적할 수 있다.